# 👕 Virtual Try-On System — Computer Vision Exam (Module 15)

**Pipeline:** Upload Image → Segment Body Part (Upper/Lower) → AI Inpainting (Segmind API) → Show Result

**Approach summary**
1. **Segmentation** — I use a pretrained clothes-parsing model, `mattmdjaga/segformer_b2_clothes` (SegFormer fine-tuned on the ATR clothing dataset), loaded via Hugging Face `transformers.pipeline("image-segmentation")`. It directly outputs per-pixel labels such as *Upper-clothes*, *Dress*, *Pants*, *Skirt*, *Belt*, etc. I combine the relevant labels into a single binary mask depending on whether the user picked **Upper** or **Lower** body.
2. **Inpainting** — The original image + binary mask + a text prompt (e.g. *"black leather jacket"*) are sent to **Segmind's Stable Diffusion 1.5 Inpainting API** (`https://api.segmind.com/v1/sd1.5-inpainting`). White mask regions are regenerated by the diffusion model according to the prompt; black regions are preserved.
3. **UI** — Built with **Gradio** (image upload, Upper/Lower radio selector, prompt textbox, Try On button, output image).

> ⚠️ You need a **free Segmind API key** — sign up at https://www.segmind.com , copy your key, and paste it into the `SEGMIND_API_KEY` cell below.

## 1. Install dependencies

In [ ]:
!pip install -q transformers torch torchvision pillow numpy opencv-python-headless gradio requests accelerate

## 2. Imports & Segmentation Model

In [ ]:
import io
import base64
import numpy as np
import requests
from PIL import Image, ImageFilter
import cv2
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1
print("Using device:", "GPU" if device == 0 else "CPU")

# Pretrained human clothes-parsing model (SegFormer fine-tuned on ATR dataset)
segmenter = pipeline(
    "image-segmentation",
    model="mattmdjaga/segformer_b2_clothes",
    device=device,
)

# Labels this model can output (ATR dataset)
# 0 Background, 1 Hat, 2 Hair, 3 Sunglasses, 4 Upper-clothes, 5 Skirt, 6 Pants,
# 7 Dress, 8 Belt, 9 Left-shoe, 10 Right-shoe, 11 Face, 12 Left-leg, 13 Right-leg,
# 14 Left-arm, 15 Right-arm, 16 Bag, 17 Scarf

UPPER_LABELS = {"Upper-clothes", "Dress", "Scarf"}
LOWER_LABELS = {"Pants", "Skirt", "Dress"}


## 3. Preprocessing

Resize the uploaded image to a manageable size (long side capped at 768px, multiple of 8 — required by most diffusion inpainting APIs) and make sure it's RGB.

In [ ]:
def preprocess_image(image: Image.Image, max_side: int = 768) -> Image.Image:
    """Resize (keeping aspect ratio) and convert to RGB. Dimensions rounded to
    the nearest multiple of 8, which most diffusion models require."""
    image = image.convert("RGB")
    w, h = image.size
    scale = max_side / max(w, h)
    if scale < 1:
        w, h = int(w * scale), int(h * scale)
    # round to multiple of 8
    w = max(8, (w // 8) * 8)
    h = max(8, (h // 8) * 8)
    return image.resize((w, h), Image.LANCZOS)


## 4. Segmentation Logic

Run the clothes-parsing model, pick out the masks for the labels relevant to
**Upper** or **Lower** body, merge them into one binary mask, then dilate + blur
the edges slightly so the inpainting blends naturally into the surrounding image.

In [ ]:
def get_clothing_mask(image: Image.Image, part: str) -> Image.Image:
    """
    part: "Upper" or "Lower"
    Returns a single-channel (L) PIL mask image, same size as `image`,
    where white (255) = region to inpaint, black (0) = region to keep.
    """
    target_labels = UPPER_LABELS if part == "Upper" else LOWER_LABELS

    results = segmenter(image)  # list of {"label": str, "mask": PIL.Image (L mode)}

    mask_arr = np.zeros((image.size[1], image.size[0]), dtype=np.uint8)
    found_any = False
    for r in results:
        if r["label"] in target_labels:
            found_any = True
            m = np.array(r["mask"].resize(image.size))
            mask_arr = np.maximum(mask_arr, m)

    if not found_any:
        raise ValueError(
            f"No '{part}' clothing region was detected in this image. "
            "Try a clearer, front-facing full/half body photo."
        )

    # Clean up the mask: dilate slightly so we cover clothing edges/shadows,
    # then feather with a Gaussian blur for a smoother inpainting blend.
    kernel = np.ones((9, 9), np.uint8)
    mask_arr = cv2.dilate(mask_arr, kernel, iterations=1)
    mask_arr = np.where(mask_arr > 30, 255, 0).astype(np.uint8)

    mask_img = Image.fromarray(mask_arr, mode="L")
    mask_img = mask_img.filter(ImageFilter.GaussianBlur(radius=3))
    return mask_img


## 5. Inpainting via Segmind API

Sends the base image, mask, and a text prompt describing the *new* clothing
to Segmind's `sd1.5-inpainting` endpoint. Images are base64-encoded so no
public hosting is required.

In [ ]:
SEGMIND_API_KEY = "PASTE_YOUR_SEGMIND_API_KEY_HERE"  # <-- get one free at https://www.segmind.com

SEGMIND_URL = "https://api.segmind.com/v1/sd1.5-inpainting"


def image_to_base64(img: Image.Image, fmt: str = "PNG") -> str:
    buf = io.BytesIO()
    img.save(buf, format=fmt)
    return base64.b64encode(buf.getvalue()).decode("utf-8")


def inpaint_clothing(image: Image.Image, mask: Image.Image, prompt: str,
                      api_key: str, negative_prompt: str = "deformed, disfigured, blurry, low quality, extra limbs") -> Image.Image:
    if not api_key or api_key == "PASTE_YOUR_SEGMIND_API_KEY_HERE":
        raise ValueError("Please set your SEGMIND_API_KEY first (sign up free at segmind.com).")

    w, h = image.size
    payload = {
        "prompt": prompt,
        "negative_prompt": negative_prompt,
        "image": image_to_base64(image),
        "mask": image_to_base64(mask),
        "samples": 1,
        "scheduler": "DDIM",
        "num_inference_steps": 30,
        "guidance_scale": 7.5,
        "strength": 1,
        "seed": 42,
        "img_width": w,
        "img_height": h,
        "base64": True,
    }
    headers = {"x-api-key": api_key, "Content-Type": "application/json"}

    response = requests.post(SEGMIND_URL, headers=headers, json=payload, timeout=120)
    if response.status_code != 200:
        raise RuntimeError(f"Segmind API error {response.status_code}: {response.text}")

    # Response is raw image bytes (JPEG) when base64=False, or JSON with base64 image
    content_type = response.headers.get("content-type", "")
    if "application/json" in content_type:
        data = response.json()
        img_b64 = data.get("image") or data["images"][0]
        img_bytes = base64.b64decode(img_b64)
    else:
        img_bytes = response.content

    return Image.open(io.BytesIO(img_bytes)).convert("RGB")


## 6. End-to-End Pipeline

In [ ]:
def virtual_try_on(image: Image.Image, part: str, prompt: str, api_key: str = None):
    api_key = api_key or SEGMIND_API_KEY
    image = preprocess_image(image)
    mask = get_clothing_mask(image, part)
    result = inpaint_clothing(image, mask, prompt, api_key)
    return image, mask, result


## 7. Quick test (optional — run after setting your API key and uploading a sample image to Colab)

In [ ]:
# from google.colab import files
# uploaded = files.upload()
# test_path = list(uploaded.keys())[0]
# test_img = Image.open(test_path)
# orig, mask, out = virtual_try_on(test_img, "Upper", "a red hooded sweatshirt")
# display(orig); display(mask); display(out)


## 8. Gradio UI

In [ ]:
import gradio as gr

def gradio_try_on(image, part, prompt, api_key_input):
    key = api_key_input.strip() if api_key_input and api_key_input.strip() else SEGMIND_API_KEY
    try:
        _, mask, result = virtual_try_on(image, part, prompt, key)
        return result, mask
    except Exception as e:
        raise gr.Error(str(e))


with gr.Blocks(title="Virtual Try-On") as demo:
    gr.Markdown("# 👕 Virtual Try-On System\nUpload a photo, choose Upper or Lower body, describe the new clothing, and click **Try On**.")
    with gr.Row():
        with gr.Column():
            image_in = gr.Image(type="pil", label="Upload Image")
            part_in = gr.Radio(["Upper", "Lower"], value="Upper", label="Body Part")
            prompt_in = gr.Textbox(label="Describe new clothing", placeholder="e.g. black leather jacket / blue denim jeans")
            key_in = gr.Textbox(label="Segmind API key (optional if set above)", type="password")
            btn = gr.Button("Try On", variant="primary")
        with gr.Column():
            result_out = gr.Image(label="Result")
            mask_out = gr.Image(label="Segmentation Mask (debug view)")

    btn.click(gradio_try_on, inputs=[image_in, part_in, prompt_in, key_in], outputs=[result_out, mask_out])

demo.launch(share=True, debug=True)


## 📋 Explanation of Approach (for submission)

| Stage | Method | Notes |
|---|---|---|
| Preprocessing | PIL resize, long side ≤768px, dims rounded to multiple of 8 | Keeps API calls fast and satisfies diffusion model size constraints |
| Segmentation | `mattmdjaga/segformer_b2_clothes` (SegFormer, HF `image-segmentation` pipeline) | Directly predicts clothing-specific labels (Upper-clothes, Pants, Skirt, Dress...), so no manual pose-based heuristics are needed |
| Mask post-processing | OpenCV dilation + Gaussian blur | Slightly expands the mask to cover clothing edges/shadows and feathers borders for a seamless inpaint blend |
| Inpainting | Segmind `sd1.5-inpainting` REST API | Images sent as base64; prompt describes the desired new garment; mask marks the region to regenerate |
| UI | Gradio `Blocks` | Upload, Upper/Lower selector, prompt box, Try On button, result + mask preview |

**Bonus features implemented:** prompt-based editing (any garment description), mask preview for transparency/debuggability, dilation+feathering for better blend quality, backend (pipeline functions) cleanly separated from the UI layer (Gradio callback only wires inputs/outputs).

**Limitations / things to improve with more time:** the SegFormer clothes model can miss loose/baggy garments in unusual poses; a SAM-based refinement step or DensePose could tighten the mask further; higher-resolution runs would need a higher-tier inpainting model (e.g. Flux Fill Pro) for photorealism.